# Smart Women Safety Platform

**ML Model & Risk Prediction**

*This notebook implements the transparent composite risk model from the research paper "Development of a Smart Women Safety Platform with Dynamic Area Risk Prediction and Safe Route Recommendation" and trains a data-driven ML model to predict area risk from environmental features matching the synthesized data summary provided.*

In [1]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score, classification_report

# Match data summary: n=1446, features with described correlations
n = 1446
np.random.seed(42)

# Street Lighting (L): mean 5.53, range 1-10, strong negative corr -0.607 with risk
L = np.random.normal(5.53, 2.5, n).clip(1, 10)

# Known Dark Spot: 25.8% True (373/1446)
dark_spot = np.random.choice([0, 1], n, p=[0.742, 0.258])

# Crowd Density: Low(40%), Medium(40%), High(20%) as per summary balance
crowd = np.random.choice([0, 1, 2], n, p=[0.4, 0.4, 0.2])  # 0=Low, 1=Med, 2=High

# Police proximity (km): mean 2.60, range 0.2-5.0, negative corr -0.203
police_dist = np.random.uniform(0.2, 5.0, n)

# Time slot for temporal multiplier: 0=Morning, 1=Afternoon, 2=Evening, 3=Night
time_slot = np.random.choice([0, 1, 2, 3], n)

# Synthesize Crime intensity proxy Cl [0,1] inversely related to lighting & dark spots
Cl = 0.6 * (1 - (L - 1) / 9) + 0.4 * dark_spot + np.random.normal(0, 0.05, n)
Cl = np.clip(Cl, 0, 1)

# Normalized lighting availability Ll [0,1]: (L-1)/9
Ll = (L - 1) / 9

# Normalized crowd/activity Hl [0,1]: 0, 0.5, 1 for Low/Med/High
Hl = np.where(crowd == 0, 0, np.where(crowd == 1, 0.5, 1.0))

# Temporal multipliers from paper (Eq 6): 0.80 (06-12), 1.00 (12-18), 1.15 (18-21), 1.25 (21-06)
multiplier = np.select(
    [time_slot == 0, time_slot == 1, time_slot == 2, time_slot == 3],
    [0.80, 1.00, 1.15, 1.25],
    default=1.0
)

# Eq 5: Base risk on 0-10 scale (corrects scale mismatch via normalization)
R_base = 10 * (0.50 * Cl + 0.30 * (1 - Ll) + 0.20 * (1 - Hl))

# Eq 6: Apply time-dependent multiplier, capped at 10
R = np.minimum(10, multiplier * R_base)

# DataFrame matching the project's synthesized dataset
df = pd.DataFrame({
    'Street_Lighting': L,
    'Known_Dark_Spot': dark_spot,
    'Crowd_Density': crowd,
    'Police_Distance_km': police_dist,
    'Time_Slot': time_slot,
    'Risk_Score': R,
    'Cl': Cl,
    'Ll': Ll,
    'Hl': Hl
})

# Save for later cells
df.to_pickle('safety_data.pkl')

# Quick EDA display (optional, won't output in script)
# display(df.head(), df.describe())

In [2]:
# --- Transparent Composite Risk Model Implementing Eqs 5 & 6 ---

def normalize_crime(crime_val):
    '''Normalize raw crime count/severity to [0,1] proportion.'''
    return np.clip(crime_val, 0, 1)

def compute_base_risk(Cl, Ll, Hl):
    '''Eq 5: Base risk score on 0-10 scale.
     Correctly normalizes crime to environmental dimensions.
    '''
    return 10 * (0.50 * Cl + 0.30 * (1 - Ll) + 0.20 * (1 - Hl))

def compute_temporal_risk(R_base, hour):
    '''Eq 6: Apply time-of-day multiplier.
    Prototype multipliers: 0.80 (06:00-12:00), 1.00 (12:00-18:00),
    1.15 (18:00-21:00), 1.25 (21:00-06:00).
    '''
    if 6 <= hour < 12: m = 0.80
    elif 12 <= hour < 18: m = 1.00
    elif 18 <= hour < 21: m = 1.15
    else: m = 1.25  # 21:00-06:00 night window
    return min(10, m * R_base)

# --- Example Usage from Paper (Section 6.1) ---
L = 5.53  # normalized lighting ~5.5/10
Ll = (L - 1) / 9
Cl = 0.52
Hl = 0.60
R_base = compute_base_risk(Cl, Ll, Hl)
R_night = compute_temporal_risk(R_base, 22)
print(f'Paper Example: Base={R_base:.2f}, At 22:00={R_night:.2f}')
print(f'Bands: <3=Green, 3-6=Yellow, >=6=Red')
print(f'Classification: {'' if R_night < 3 else 'Yellow' if R_night < 6 else 'Red'} (risk band)')

Paper Example: Base=4.89, At 22:00=6.11
Bands: <3=Green, 3-6=Yellow, >=6=Red
Classification: Red (risk band)


In [3]:
# --- Train Data-Driven ML Model ---
# Trains a RandomForest regressor to predict risk score from features,
# allowing comparison/composite with the transparent paper model.
X = df[['Street_Lighting', 'Known_Dark_Spot', 'Crowd_Density', 'Police_Distance_km', 'Time_Slot']]
y = df['Risk_Score']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
rf = RandomForestRegressor(n_estimators=300, random_state=42, max_depth=15)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print(f'RandomForest Performance: MAE={mae:.2f}, R²={r2:.3f}')

# Feature Importances (should highlight Street Lighting as strongest negative correlate)
importances = pd.Series(rf.feature_importances_, X.columns).sort_values(ascending=False)
print('\nFeature Importances (higher = more influential on risk prediction):')
print(importances)

# Predict risk for a 'safe-profile' locality: well-lit, no dark spot, medium crowd, close police
sample = pd.DataFrame({'Street_Lighting': [7.0], 'Known_Dark_Spot': [0], 'Crowd_Density': [1], 'Police_Distance_km': [1.5], 'Time_Slot': [1]})
pred = rf.predict(sample)[0]
band = 'Green' if pred < 3 else 'Yellow' if pred < 6 else 'Red'
print(f'\nML Prediction for safe-profile locality: Risk Score={pred:.2f}, Band={band}')

RandomForest Performance: MAE=0.26, R²=0.973

Feature Importances (higher = more influential on risk prediction):
Street_Lighting       0.596830
Known_Dark_Spot       0.145177
Time_Slot             0.141279
Crowd_Density         0.105878
Police_Distance_km    0.010837
dtype: float64

ML Prediction for safe-profile locality: Risk Score=2.86, Band=Green


In [4]:
# --- Risk Band Assignment (Paper Thresholds) ---
def risk_band(score):
    if score < 3: return 'Green (Lower estimated exposure)'
    elif score < 6: return 'Yellow (Moderate estimated exposure)'
    else: return 'Red (Higher estimated exposure)'

# --- Confidence Label (Paper Eq 7: ql = min(1, n_locality/30)) ---
def confidence_label(n_incidents):
    ql = min(1, n_incidents / 30)
    if ql >= 0.8: return 'High evidence coverage'
    elif ql >= 0.5: return 'Moderate evidence coverage'
    else: return 'Sparse evidence coverage'

# --- Compare Paper Model vs ML Model ---
print('\n=== Paper Model Example (Section 6.1) ===')
Cl, Ll, Hl = 0.52, 0.55, 0.40
R_base = compute_base_risk(Cl, Ll, Hl)
R_night = compute_temporal_risk(R_base, 22)
print(f'Base risk: {R_base:.2f}, Night 22:00 risk: {R_night:.2f}')
print(f'Risk band: {risk_band(R_night)}')

print('\n=== ML Model Prediction ===')
sample_df = pd.DataFrame({'Street_Lighting': [5.5], 'Known_Dark_Spot': [1], 'Crowd_Density': [1], 'Police_Distance_km': [2.8], 'Time_Slot': [3]})
pred = rf.predict(sample_df)[0]
print(f'Predicted risk score: {pred:.2f}')
print(f'Risk band: {risk_band(pred)}')
print(f'Confidence: {confidence_label(30)}')  # assuming ~30 localities

# --- Eq 9-12: Route metrics demonstration ---
print('\n=== Route Exposure Formulas (Paper Sec 7.2-7.3) ===')
def path_exposure(risk_vals, dist_vals):
    '''Mean risk (Eq 9): R(P) = sum(Re*de) / sum(de)'''
    return sum(r * d for r, d in zip(risk_vals, dist_vals)) / sum(dist_vals)

def path_uncertainty(risk_vals, dist_vals, conf_vals):
    '''Uncertainty U(P) (Eq 11): sum((1-qe)*de) / sum(de)'''
    return sum((1 - c) * d for c, d in zip(conf_vals, dist_vals)) / sum(dist_vals)

# Example route segment
risks = [4.2, 5.1, 3.8]
dists = [5.0, 3.2, 4.5]
conf = [0.9, 0.7, 0.85]
print(f'Mean exposure: {path_exposure(risks, dists):.2f}')
print(f'Uncertainty: {path_uncertainty(risks, dists, conf):.2f}')


=== Paper Model Example (Section 6.1) ===
Base risk: 5.15, Night 22:00 risk: 6.44
Risk band: Red (Higher estimated exposure)

=== ML Model Prediction ===
Predicted risk score: 7.63
Risk band: Red (Higher estimated exposure)
Confidence: High evidence coverage

=== Route Exposure Formulas (Paper Sec 7.2-7.3) ===
Mean exposure: 4.29
Uncertainty: 0.17
